<a href="https://colab.research.google.com/github/ardianita/data-science-2026/blob/main/Pertemuan_10_Ardianita_Fauziyah_250401020128.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama : Ardianita Fauziyah

NIM : 250401020128

Kelas : IF403

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/Nas-virat/Telco-Customer-Churn/master/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(df.shape)
print(df["Churn"].value_counts(normalize=True))
df

(7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [ ]:
from sklearn.model_selection import train_test_split

# Hilangkan spasi di awal/akhir
df["TotalCharges"] = df["TotalCharges"].str.strip()

# Ubah string kosong menjadi NaN
df["TotalCharges"] = df["TotalCharges"].replace("", pd.NA)

# Konversi ke numerik
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Isi nilai yang kosong
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())


df = df.drop(columns=['customerID'])

# Encode target
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

df = pd.get_dummies(
    df,
    columns=[
        'gender',
        'Partner',
        'Dependents',
        'PhoneService',
        'MultipleLines',
        'InternetService',
        'OnlineSecurity',
        'OnlineBackup',
        'DeviceProtection',
        'TechSupport',
        'StreamingTV',
        'StreamingMovies',
        'Contract',
        'PaperlessBilling',
        'PaymentMethod'
    ],
    drop_first=True,
    dtype=int
)

# Pisahkan fitur dan target
x = df.drop('Churn', axis=1)   # drop kolom yang ingin diprediksi
y = df['Churn']  # kolom yang ingin diprediksi

# Train-Test Split
x_training, x_test, y_training, y_test = train_test_split(
    x,
    y,
    test_size=0.2,  # membagi data menjadi 20% test dan 80% training
    stratify=y,
    random_state=42
)
print(f'Data Training: {x_training.shape[0]} baris, Data Test: {x_test.shape[0]} baris')

Data Training: 5634 baris, Data Test: 1409 baris


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",   # menangani data tak seimbang
    random_state=42)

rf.fit(x_training, y_training)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Prediksi
y_pred = rf.predict(x_test)
y_prob = rf.predict_proba(x_test)[:, 1]

# Metrik
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Precision : 0.6296
Recall    : 0.5000
F1-Score  : 0.5574
ROC-AUC   : 0.8246

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


Confusion Matrix
[[925 110]
 [187 187]]


In [ ]:
import pandas as pd

# Prediksi probabilitas churn (kelas 1)
y_prob = rf.predict_proba(x_test)[:, 1]

# Tampilkan hasil
hasil = pd.DataFrame({
    'Aktual': y_test.values,
    'Prediksi': y_pred,
    'Probabilitas_Churn': y_prob
})

print(hasil.head(10))

   Aktual  Prediksi  Probabilitas_Churn
0       0         0            0.000000
1       0         1            0.786667
2       0         0            0.090000
3       0         0            0.280000
4       0         0            0.000000
5       0         0            0.416667
6       0         0            0.393333
7       0         0            0.110000
8       0         0            0.006667
9       1         0            0.460000


Model Random Forest dapat menghasilkan probabilitas churn untuk setiap pelanggan melalui predict_proba(), sehingga pelanggan dapat diurutkan berdasarkan tingkat risiko churn. Nilai Precision sebesar **0,6296**, Recall sebesar **0,5000**, dan F1-Score sebesar **0,5574** menunjukkan bahwa model cukup baik dalam mengidentifikasi pelanggan yang berpotensi churn, walaupun masih ada beberapa pelanggan churn yang belum berhasil diprediksi. Sementara itu, nilai ROC-AUC sebesar **0,8246** menunjukkan bahwa model memiliki kemampuan yang baik dalam membedakan pelanggan yang churn.

**Kesimpulan**

* Hal yang dapat saya pelajari dari praktikum ini adalah mengenai bagaimana cara melakukan pembersihan data dan penanganan nilai hilang pada kolom TotalCharges, melakukan encoding variabel kategorikal menggunakan One-Hot Encoding, menerapkan stratified train-test split (80% training / 20% testing) pada dataset berlabel tak seimbang (imbalanced dataset), membangun model Random Forest Classifier, serta mengevaluasi performa model menggunakan metrik Precision, Recall, F1-Score, ROC-AUC, dan analisis Confusion Matrix.

* Pada praktikum ini, saya menemukan bahwa analisis awal menunjukkan adanya ketidakseimbangan kelas target, yaitu 73,5% pelanggan No Churn dan 26,5% pelanggan Churn. Model Random Forest yang dibangun berhasil memprediksi probabilitas churn untuk tiap pelanggan dan mencatatkan nilai ROC-AUC sebesar 0,8246, yang menunjukkan kemampuan pembedaan yang sangat baik dalam memisahkan pelanggan churn dan non-churn.

* Selain itu, saya juga menemukan sebuah keterbatasan. Keterbatasan tersebut terletak pada nilai Recall sebesar 0,5000 dan F1-Score sebesar 0,5574 (dengan Precision 0,6296). Hal ini menandakan bahwa model masih melewatkan sekitar 50% dari total pelanggan yang sebenarnya mengalami churn (False Negative sebanyak 187 dari 374 data uji). Penanganan data tidak seimbang hanya dengan parameter class_weight='balanced' belum sepenuhnya optimal, sehingga masih diperlukan penelitian lanjutan untuk meningkatkan sensitivitas deteksi churn.